# Scalar conservation law: Burgers

In [ ]:
#    APM41012EP course notebook - Chapter 11 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Nonlinear scalar conservation law: Burgers
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = "seaborn"

## Burgers equation

The variable $u(t,x)$, $x\in\mathbb{R}$,  $t\in\mathbb{R}_+$, satisfies the partial differential equation:

$$\partial_t u + \partial_x(f(u)) = 0 $$

with the nonlinear flux:

$$f(u) = \frac{u^2}{2},$$

and the Cauchy problem is completed by the initial data:

$$u(0,x) = u_0(x).$$

In this notebook, we consider the little hat problem studied in the lecture notes, as well as a square wave, which allows us to see the entropic weak solution of the two Riemann problems: with $U_g<U_d$, made of a rarefaction wave, and with $U_g>U_d$, made of a shock wave.

In [ ]:
# flux computation
def f(x):
    return (0.5*x*x)

In [ ]:
# Little hat initial data
def hat_ini(x):
    u = np.empty(x.size)
    for ix, xi in enumerate(x):
        if xi <= -1 or xi>=1:
            u[ix] = 0
        elif xi < 0:
            u[ix] = xi + 1
        else:
            u[ix] = -xi + 1
    return u

# little hat solution for the Burgers equation
def compute_uexa_hat(xini, xend, t):
    u = np.empty(5)
    u[0] = 0
    u[1] = 0
    u[2] = np.sqrt(2/(1+t))
    u[3] = 0
    u[4] = 0
    
    x = np.empty(5)
    x[0] = xini
    x[1] = -1
    x[2] = np.sqrt(2*(1+t)) - 1
    x[3] = np.sqrt(2*(1+t)) - 1
    x[4] = xend

    return x, u 

In [ ]:
# solution of the Riemann problem at x/t = 0 for two states Ug and Ud on the left and on the right - scalar version
# Denoted W_R(x/t=0,Ug,Ud) in the written notes and the lecture slides
#
def Riemann(Ug,Ud):
    Riem = Ug
    if (max(Ug,Ud)<=0) : Riem = Ud
    else :
        if Ug<=0 and Ud>=0 :
            Riem=0
        else :
            if Ug+Ud<=0 : Riem = Ud
    return Riem
#
# It is possible to vectorize, inefficiently,
# with the np.vectorize function
# 
Riemann_vec = np.vectorize(Riemann)

# More efficient vectorized version with np.where
# used in the Godunov scheme below, allowing a 
# solution with several thousand cells in a few seconds
#
def Riemann2(Ug,Ud):
    Riem = Ug
    c = np.stack((Ug, Ud))
    Riem = np.where(np.amax(c,axis=0)<=0, Ud, Riem)
    Riem = np.where((Ug<=0) & (Ud>=0), 0, Riem)
    Riem = np.where((Ug>=0) & (Ud<=0) & ((Ug+Ud)<0), Ud, Riem)
    return Riem

### Godunov scheme

The Godunov scheme for the Burgers equation is used here to solve, first, the little hat problem. We saw in the course that the scheme, under the CFL condition, converges to the unique entropic weak solution of the Cauchy problem.

A slider shows the influence of the discretization and the ability of the scheme to converge to the unique entropic weak solution studied in the lecture notes. We compare the solution computed by the scheme with the exact solution predicted by the theory, and one can also provide a convergence curve for the scheme. 

In [ ]:
def plot_sol(xmin, xmax, nx, tini, tend, cfl, a, uini):
    
    fig = go.Figure()
    nxplot = 2000
    dxplot = (xmax-xmin) / nxplot
    xplot = np.linspace(xmin+(dxplot/2), xmax-(dxplot/2), nxplot)
    fig.add_trace(go.Scatter(x=xplot, y=uini(xplot), name='initial sol.', line_color='rgb(140,140,140)'))

    xexa10, uexa10 = compute_uexa_hat(xmin, xmax, tend/2)
    xexa15, uexa15 = compute_uexa_hat(xmin, xmax, 3*tend/4)
    xexa20, uexa20 = compute_uexa_hat(xmin, xmax, tend)

    fig.add_trace(go.Scatter(x=xexa10, y=uexa10, mode='lines', name='exact sol. at t=0.5*tend', 
                             line=dict(dash='dash', color='rgb(76,114,176)')))
    fig.add_trace(go.Scatter(x=xexa15, y=uexa15, mode='lines', name='exact sol. at t=0.75*tend', 
                             line=dict(dash='dash', color='rgb(221,132,82)')))
    fig.add_trace(go.Scatter(x=xexa20, y=uexa20, mode='lines', name='exact sol. at t=tend', 
                             line=dict(dash='dash', color='rgb(85,168,104)')))

    for nx_i in nx:

        dx = (xmax-xmin) / nx_i
        x = np.linspace(xmin+dx/2, xmax-dx/2, nx_i)

        u = np.zeros(nx_i+1)
        u[1:] = uini(x)
        u[0] = u[-1]

        nt = int(a*(tend-tini)/(cfl*dx)) + 1
        dt = (tend-tini)/(nt-1)
        #print(nt, dt)
        dtoverdx = dt/dx

        u10 = np.zeros(nx_i)
        u15 = np.zeros(nx_i)
        flux = np.zeros(nx_i)
        etats = np.zeros(nx_i)

        for it in range(nt-1):

            etats[:-1] = Riemann2(u[1:-1],u[2:])
            etats[-1] = Riemann(u[-1],u[0])
            flux = f(etats)

            u[2:] += -dtoverdx * (flux[1:] - flux[0:-1])
            u[1] += -dtoverdx * (flux[0] - flux[-1])
            u[0] = u[-1] #  periodic boundary conditions

            t = (it+1)*dt
            if it == nt//2: u10[:] = u[1:]
            if it == 3*(nt//4): u15[:] = u[1:]

        fig.add_trace(go.Scatter(x=x, y=u10, mode='lines', name='num. sol. at t=0.5*tend', 
                                  line_color='rgb(76,114,176)', opacity=0.75, visible=False))
        fig.add_trace(go.Scatter(x=x, y=u15, mode='lines', name='num. sol. at t=0.75*tend', 
                                 line_color='rgb(221,132,82)', opacity=0.75, visible=False))
        fig.add_trace(go.Scatter(x=x, y=u[1:], mode='lines', name='num. sol. at t=tend', 
                                 line_color='rgb(85,168,104)', opacity=0.75, visible=False))

    fig.data[4].visible = True  
    fig.data[5].visible = True  
    fig.data[6].visible = True

    # Create and add slider
    steps = []
    for i, nx_i in enumerate(nx):
        args = [{"visible": [((el>=0 and el<=3)or (el>=3*i+4 and el<=3*i+6)) for el in range(len(fig.data))]}]
        ##print(args)
        step = dict(method="update", label = f"{nx_i}", args=args)
        steps.append(step)
    sliders = [dict(currentvalue={'prefix': 'nx = '}, steps=steps)]

    fig.update_layout(sliders=sliders, title=f"Godunov scheme: cfl={cfl}", height=500, 
                      legend=dict(x=0.75, y=0.95, bgcolor='rgba(0,0,0,0)'))

    fig.show()

In [ ]:
a = 1.

xmin = -2.0
xmax = 4.0
nx = [125, 250, 500, 1000, 2000]

tini = 0.
tend = 6.
# Do not increase tend beyond 11 because we have used periodic boundary conditions,
# but the exact solution is not computed taking this type of boundary conditions into account.

uini = hat_ini

cfl = 0.95

plot_sol(xmin, xmax, nx, tini, tend, cfl, a, hat_ini)